# Checkpoint 1 — Dataset Integrity Audit

## Objective
This notebook conducts a structural and reproducible integrity audit of the 9 raw CSV tables from the Brazilian E-Commerce Public Dataset by Olist.

In accordance with project guidelines for Checkpoint 1:
- Performs structural validation only (dimensions, inferred dtypes, null percentages, duplicate rows, temporal bounds, and structural column presence).
- Identifies candidate identifier columns strictly from schema naming conventions.
- Does not perform customer identity validation, RFM segmentation, churn modeling, or feature engineering.
- All audit logic is implemented in `src/audit.py`, keeping this notebook focused on reporting and visualization.

## 1. Setup and Environment Configuration

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.audit import run_audit, get_checkpoints_dir

print(f"Project root resolved: {project_root}")

## 2. Execute Dataset Integrity Audit

Run `run_audit()` from `src.audit` to perform the read-only audit, verify raw file immutability, and generate `data/checkpoints/01_integrity.json`.

In [ ]:
# Execute audit
checkpoint_data = run_audit(project_root)
summary = checkpoint_data["summary"]

print(f"Audit Timestamp (UTC): {checkpoint_data['audit_timestamp_utc']}")
print(f"Total tables audited: {summary['total_tables_audited']}")
print(f"Total rows audited: {summary['total_row_count']:,}")
print(f"Raw data immutability verified: {summary['raw_data_immutability_verified']}")
print(f"All structural columns present: {summary['all_structural_columns_present']}")

## 3. Table Overview: Dimensions and Duplicate Counts

In [ ]:
table_records = []
for tname, tdata in checkpoint_data["tables"].items():
    total_missing = sum(tdata["missing_counts"].values())
    table_records.append({
        "Table Name": tname,
        "File Name": tdata["filename"],
        "Row Count": tdata["row_count"],
        "Column Count": tdata["column_count"],
        "Duplicate Rows": tdata["duplicate_rows"],
        "Total Missing Values": total_missing,
        "Candidate Identifiers": ", ".join(tdata["candidate_identifier_columns"])
    })

df_overview = pd.DataFrame(table_records)
df_overview.sort_values(by="Row Count", ascending=False).reset_index(drop=True)

## 4. Missing Value Analysis by Column

Inspect columns across all tables that have non-zero missing values.

In [ ]:
missing_records = []
for tname, tdata in checkpoint_data["tables"].items():
    for col, count in tdata["missing_counts"].items():
        if count > 0:
            missing_records.append({
                "Table": tname,
                "Column": col,
                "Inferred Dtype": tdata["dtypes"][col],
                "Missing Count": count,
                "Missing %": tdata["missing_percentages"][col]
            })

df_missing = pd.DataFrame(missing_records)
if not df_missing.empty:
    df_missing = df_missing.sort_values(by="Missing %", ascending=False).reset_index(drop=True)
df_missing

## 5. Datetime Column Audit & Temporal Bounds

Check temporal bounds (`min_valid_date` to `max_valid_date`) and unparseable counts for all detected date/time columns.

In [ ]:
datetime_records = []
for tname, tdata in checkpoint_data["tables"].items():
    for col, dt_stats in tdata["datetime_columns"].items():
        datetime_records.append({
            "Table": tname,
            "Datetime Column": col,
            "Non-Null Raw": dt_stats["total_non_null_raw"],
            "Valid Parsed": dt_stats["valid_parsed_count"],
            "Unparseable": dt_stats["unparseable_count"],
            "Min Valid Date": dt_stats["min_valid_date"],
            "Max Valid Date": dt_stats["max_valid_date"]
        })

df_datetime = pd.DataFrame(datetime_records)
df_datetime.sort_values(by=["Table", "Datetime Column"]).reset_index(drop=True)

## 6. Structural Column Presence Across Related Tables

Verify the presence of matching identifier column names between tables without asserting primary/foreign key semantics.

In [ ]:
df_structural = pd.DataFrame(checkpoint_data["structural_column_presence_checks"])
df_structural[["relationship_label", "table_a", "column_a_present", "table_b", "column_b_present", "both_columns_present"]]

## 7. Checkpoint 1 Summary

### Data Analysis Key Findings
- **Dataset Dimensions**: All 9 raw CSV files are present and verified. The total dataset spans 1,550,922 rows across 9 tables.
- **Duplicates**: Only `olist_geolocation_dataset` exhibits duplicate rows (261,831 duplicate rows out of 1,000,163 total rows, or ~26.18%). All other 8 tables have exactly 0 duplicate rows.
- **Missing Values**: Columns with missing values include review comments (`review_comment_message` at 58.70% missing, `review_comment_title` at 88.34% missing), order delivery milestone timestamps (`order_delivered_customer_date` with 2,965 nulls, `order_delivered_carrier_date` with 1,783 nulls, `order_approved_at` with 160 nulls), and product specifications (610 nulls in category name and dimensions).
- **Datetime Integrity**: All 8 datetime columns parsed with 0 unparseable values. Valid date ranges span from September 2016 through October/November 2018.
- **Structural Presence**: All 9 expected column relationships between tables were confirmed structurally present.
- **Raw Data Immutability**: All 9 raw files maintained identical SHA-256 hashes pre- and post-audit.

### Insights or Next Steps
- Checkpoint 1 structural validation is complete and aggregate statistics are preserved in `data/checkpoints/01_integrity.json` without any row-level or identifying data.
- Awaiting user review and authorization before proceeding to Checkpoint 2 (Customer Identity Validation).